# What a Weaviate group-by does and does not return

Creates a small collection, inserts full objects, then runs the **same hybrid query twice** —
once ungrouped and once grouped — asking both times for score, explain score, certainty,
distance and both timestamps.

The claim under test:

> An object returned inside a group carries only `distance`, `id` and `vector`.
> Score, explain score, certainty and the timestamps are not available inside a group —
> through **any** API, not just gRPC.

The last cells prove *why* from the server's own GraphQL schema, which is more convincing than
any client's behaviour: a client that shows nothing might just be dropping it.

```
uv run --with weaviate-client --with jupyter jupyter lab grouped_metadata_demo.ipynb
```

Needs a Weaviate on `localhost:8080` / `50051`. No model provider or API keys —
the collection supplies its own vectors.


In [1]:
import os, json, urllib.request
import weaviate
import weaviate.classes.config as wc
from weaviate.classes.init import Auth
from weaviate.classes.query import GroupBy, MetadataQuery

HTTP_HOST = os.environ.get('WEAVIATE_HTTP_HOST', 'localhost')
HTTP_PORT = int(os.environ.get('WEAVIATE_HTTP_PORT', '8080'))
API_KEY   = os.environ.get('WEAVIATE_API_KEY', 'root-user-key')
COLLECTION = 'GroupedMetadataDemo'

client = weaviate.connect_to_custom(
    http_host=HTTP_HOST, http_port=HTTP_PORT, http_secure=False,
    grpc_host=os.environ.get('WEAVIATE_GRPC_HOST', 'localhost'),
    grpc_port=int(os.environ.get('WEAVIATE_GRPC_PORT', '50051')), grpc_secure=False,
    auth_credentials=Auth.api_key(API_KEY) if API_KEY else None,
)
print('connected:', client.get_meta()['version'])


connected: 1.39.0


## 1. A collection and some real objects

`self_provided` vectors, so nothing here depends on an embedding provider being reachable and
the numbers are reproducible. Vectors sit on one axis at `x = year - 2019`, and the index uses
squared L2 rather than the default cosine — with every vector pointing the same way, cosine
would score them all identically and the ordering would carry no information.


In [2]:
if client.collections.exists(COLLECTION):
    client.collections.delete(COLLECTION)

client.collections.create(
    COLLECTION,
    vector_config=wc.Configure.Vectors.self_provided(
        vector_index_config=wc.Configure.VectorIndex.hnsw(
            distance_metric=wc.VectorDistances.L2_SQUARED)),
    properties=[
        wc.Property(name='title',    data_type=wc.DataType.TEXT),
        # FIELD tokenization: group keys are exact values, not words.
        wc.Property(name='category', data_type=wc.DataType.TEXT,
                    tokenization=wc.Tokenization.FIELD),
        wc.Property(name='year',     data_type=wc.DataType.INT),
    ],
)

ROWS = [
    ('11111111-0000-4000-8000-000000000001', 'red admiral butterfly', 'red',   2020),
    ('11111111-0000-4000-8000-000000000002', 'red maple leaf',        'red',   2021),
    ('11111111-0000-4000-8000-000000000003', 'red brick wall',        'red',   2022),
    ('11111111-0000-4000-8000-000000000004', 'green sea turtle',      'green', 2023),
    ('11111111-0000-4000-8000-000000000005', 'green tea leaves',      'green', 2024),
    ('11111111-0000-4000-8000-000000000006', 'blue morpho wing',      'blue',  2025),
]

coll = client.collections.use(COLLECTION)
for uuid, title, category, year in ROWS:
    coll.data.insert(
        properties={'title': title, 'category': category, 'year': year},
        uuid=uuid,
        vector=[float(year - 2019), 0.0],
    )
print(f'inserted {len(coll)} objects')


inserted 6 objects


## 2. Ungrouped hybrid — ask for everything

`vector=` is supplied explicitly because the collection has no vectorizer, so the server has
nothing to embed the query text with. This is a normal hybrid otherwise: keyword half plus
vector half, fused.


In [3]:
WANT = MetadataQuery(
    score=True, explain_score=True, distance=True, certainty=True,
    creation_time=True, last_update_time=True,
)

flat = coll.query.hybrid(query='red', vector=[0.0, 0.0], limit=10, return_metadata=WANT)

print('metadata type:', type(flat.objects[0].metadata).__name__)
print('fields on it :', [f for f in dir(flat.objects[0].metadata) if not f.startswith('_')])
print()
for o in flat.objects:
    m = o.metadata
    print(f"{o.properties['title']:24s} score={m.score!s:<12} created={m.creation_time}")
print()
print('explain score for the top hit:')
print(flat.objects[0].metadata.explain_score)


metadata type: MetadataReturn
fields on it : ['certainty', 'creation_time', 'distance', 'explain_score', 'is_consistent', 'last_update_time', 'rerank_score', 'score']

red admiral butterfly    score=0.25         created=2026-08-25 19:27:26.010000+00:00
red maple leaf           score=0.25         created=2026-08-25 19:27:26.014000+00:00
red brick wall           score=0.25         created=2026-08-25 19:27:26.018000+00:00

explain score for the top hit:

Hybrid (Result Set keyword,bm25) Document 11111111-0000-4000-8000-000000000001: original score 0.6301338, normalized score: 0.25


Everything asked for came back: a fused **score**, a human-readable **explain score** showing
the keyword and vector halves separately, and **timestamps**.


## 3. The same hybrid, grouped

Identical query, identical `return_metadata`. The only difference is `group_by=`.


In [4]:
grouped = coll.query.hybrid(
    query='red', vector=[0.0, 0.0], limit=10, return_metadata=WANT,
    group_by=GroupBy(prop='category', number_of_groups=10, objects_per_group=10),
)

first = grouped.objects[0].metadata
print('metadata type:', type(first).__name__)
print('fields on it :', [f for f in dir(first) if not f.startswith('_')])
print()
for o in grouped.objects:
    print(f"{o.properties['title']:24s} group={o.belongs_to_group!r:<9} "
          f"distance={o.metadata.distance}")


metadata type: GroupByMetadataReturn
fields on it : ['distance']

red admiral butterfly    group='red'     distance=0.0
red maple leaf           group='red'     distance=0.0
red brick wall           group='red'     distance=0.0


The metadata object is a **different type** — `GroupByMetadataReturn` — and it has exactly one
field: `distance`. There is no `score` attribute to read, so this alone does not prove the
server withheld it; the client may simply have nowhere to put it. Cell 5 settles that.

Confirm the score is genuinely unreachable rather than just unprinted:


In [5]:
try:
    print(grouped.objects[0].metadata.score)
except AttributeError as e:
    print('AttributeError ->', e)

# The group record carries its own numbers, which do survive.
g = grouped.groups['red']
print()
print('group fields  :', [f for f in dir(g) if not f.startswith('_')])
print(f'group {g.name!r}: n={g.number_of_objects} min_distance={g.min_distance} '
      f'max_distance={g.max_distance} rerank_score={g.rerank_score}')


AttributeError -> 'GroupByMetadataReturn' object has no attribute 'score'

group fields  : ['max_distance', 'min_distance', 'name', 'number_of_objects', 'objects', 'rerank_score']
group 'red': n=3 min_distance=0.0 max_distance=0.0 rerank_score=0.0


## 4. Vectors and ids do survive

Worth showing, because it rules out the simpler explanation that grouping just turns metadata
off wholesale.


In [6]:
with_vectors = coll.query.hybrid(
    query='red', vector=[0.0, 0.0], limit=10, include_vector=True,
    group_by=GroupBy(prop='category', number_of_groups=10, objects_per_group=10),
)
for o in with_vectors.objects[:3]:
    print(f"{o.properties['title']:24s} uuid={str(o.uuid)[:8]}… vector={o.vector['default']}")


red admiral butterfly    uuid=11111111… vector=[1.0, 0.0]
red maple leaf           uuid=11111111… vector=[2.0, 0.0]
red brick wall           uuid=11111111… vector=[3.0, 0.0]


## 5. Why — the server's own schema

This is the part that settles it. GraphQL introspection asks the server directly what fields
exist on an object's `_additional` block, ungrouped versus inside a group. No client in the way.


In [7]:
def graphql(query: str):
    req = urllib.request.Request(
        f'http://{HTTP_HOST}:{HTTP_PORT}/v1/graphql',
        data=json.dumps({'query': query}).encode(),
        headers={'Content-Type': 'application/json',
                 'Authorization': f'Bearer {API_KEY}'},
    )
    return json.loads(urllib.request.urlopen(req).read())

def fields_of(type_name: str):
    r = graphql('{ __type(name: "%s") { fields { name } } }' % type_name)
    t = r['data']['__type']
    return sorted(f['name'] for f in t['fields']) if t else None

ungrouped = fields_of(f'{COLLECTION}Additional')
in_group  = fields_of(f'{COLLECTION}AdditionalGroupHitsAdditional')

print(f'ungrouped  _additional  ({len(ungrouped)} fields)')
for f in ungrouped:
    print('   ', f)
print()
print(f'inside a group          ({len(in_group)} fields)')
for f in in_group:
    print('   ', f)
print()
print('lost when grouping:', sorted(set(ungrouped) - set(in_group)))


ungrouped  _additional  (13 fields)
    certainty
    classification
    creationTimeUnix
    distance
    explainScore
    generate
    group
    id
    lastUpdateTimeUnix
    queryProfile
    score
    vector
    vectors

inside a group          (3 fields)
    distance
    id
    vector

lost when grouping: ['certainty', 'classification', 'creationTimeUnix', 'explainScore', 'generate', 'group', 'lastUpdateTimeUnix', 'queryProfile', 'score', 'vectors']


And asking for `score` inside a group is a **schema error**, not an empty value — the field
does not exist there at all:


In [8]:
r = graphql('''{
  Get {
    %s(
      bm25: {query: "red"}
      groupBy: {path: ["category"], groups: 10, objectsPerGroup: 10}
    ) {
      _additional { group { hits { _additional { id score explainScore } } } }
    }
  }
}''' % COLLECTION)

if 'errors' in r:
    for err in r['errors']:
        print(err['message'])
else:
    print('No error -- the server now returns scores in groups!')
    print(json.dumps(r, indent=2)[:800])


Cannot query field "score" on type "GroupedMetadataDemoAdditionalGroupHitsAdditional".
Cannot query field "explainScore" on type "GroupedMetadataDemoAdditionalGroupHitsAdditional".


## Summary

| metadata | ungrouped | inside a group |
|---|---|---|
| `id` | yes | **yes** |
| `distance` | yes | **yes** |
| `vector` | yes | **yes** |
| `score` | yes | no |
| `explainScore` | yes | no |
| `certainty` | yes | no |
| `creationTimeUnix` | yes | no |
| `lastUpdateTimeUnix` | yes | no |

So a grouped BM25 or hybrid search is **ordered by a score it will not tell you**. The ordering
is applied correctly — groups come back ranked by their best member — but the number behind it
is not retrievable, and neither is the explain-score breakdown that would justify it.

This is why the DBeaver plugin hides the `_score`, `_explainScore`, `_certainty`, `_created`
and `_updated` columns while grouping: they could never be filled.


In [9]:
client.collections.delete(COLLECTION)
client.close()
print(f'deleted {COLLECTION}, connection closed')


deleted GroupedMetadataDemo, connection closed
